# Notebook 5: Entity-Anchored Guided Clustering (Production)
---
## Strategy
Anchor topics to **high-frequency entities** (People, Places, Orgs) extracted via NER.
Uses BERTopic `seed_topic_list` to guide cluster formation around known entities.

## Flow
```
Raw Data -> NER Extraction (Person, Org, Location)
         -> Frequency Map (Top 15 entities)
         -> Guided BERTopic (seed_topic_list)
         -> Outlier Reduction (force -1 to nearest)
         -> 0% Noise Output
```

## When to Use
- For Search API integration (stable, entity-based topics)
- When you KNOW the key persons/places that matter
- Production deployment with zero noise tolerance

## Key Insight
- Person entities (Modi, Trump) take priority over Places (Iran)
- `reduce_outliers(strategy="embeddings")` forces every title into a cluster
- Result: Every document gets a topic label for your search engine


In [ ]:
!pip install -q bertopic sentence-transformers hdbscan umap-learn pandas scikit-learn transformers
print("Ready")

In [ ]:
import pandas as pd, numpy as np, re, html
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

## Step 1: NER Extraction
Extract Person, Organization, Location entities from each title.

In [ ]:
from transformers import pipeline as hf_pipeline

ner_pipe = hf_pipeline("ner", model="Babelscape/wikineural-multilingual-ner",
                       aggregation_strategy="simple")

DOMAIN_STOPS = {"abp","shorts","viral","subscribe","youtube","news","live","at2","n18v"}
def clean(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    tokens = [t for t in text.lower().split() if not (t.isascii() and len(t)<=2) and t not in DOMAIN_STOPS]
    return " ".join(tokens).strip()

# --- UNIVERSAL DATA LOADER ---
import pandas as pd
import os

# Get the file extension
file_ext = os.path.splitext(filename)[1].lower()

if file_ext == '.xlsx':
    print("📂 Detected Excel file. Using read_excel...")
    df = pd.read_excel(filename)
elif file_ext == '.csv' or file_ext == '.txt':
    print("📂 Detected Text/CSV file. Using read_csv with encoding recovery...")
    try:
        df = pd.read_csv(filename, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(filename, encoding='latin1')
        except:
            df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')
else:
    print(f"⚠️ Unknown format {file_ext}. Attempting generic read...")
    df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')

# Ensure we have a title column
if "title" not in df.columns:
    if len(df.columns) == 1:
        df.columns = ["title"]
    else:
        df.columns = ["title"] + list(df.columns[1:])
# ------------------------------
if "title" not in df.columns: df.columns=["title"]+list(df.columns[1:])
df["clean_text"] = df["title"].fillna("").astype(str).map(clean)
df = df[df["clean_text"]!=""].reset_index(drop=True)

# Extract entities
from collections import Counter
person_counter = Counter()
for i in range(0, len(df), 32):
    batch = df["clean_text"].iloc[i:i+32].tolist()
    results = ner_pipe(batch)
    for r in results:
        for ent in r:
            if ent["entity_group"] == "PER":
                person_counter[ent["word"].strip()] += 1

print("Top 15 People:")
for name, count in person_counter.most_common(15):
    print(f"  {name}: {count}")

## Step 2: Guided Clustering with Entity Seeds

In [ ]:
embedder = SentenceTransformer("sentence-transformers/LaBSE")
docs = df["clean_text"].tolist()
embeddings = embedder.encode(docs, show_progress_bar=True, batch_size=32)

# Build seed list from top entities
seeds = []
for name, count in person_counter.most_common(15):
    seeds.append(name.lower().split())

model = BERTopic(
    embedding_model=embedder,
    seed_topic_list=seeds,
    min_topic_size=15,
    nr_topics="auto",
    verbose=True
)
topics, _ = model.fit_transform(docs, embeddings)
print(f"Before outlier reduction: {len(set(topics))-1} topics, {round(100*(np.array(topics)==-1).mean(),1)}% noise")

## Step 3: Outlier Reduction (Zero Noise)
Force-assign every noise document to its nearest cluster.

In [ ]:
new_topics = model.reduce_outliers(docs, topics, strategy="embeddings")
model.update_topics(docs, topics=new_topics)
df["topic_id"] = new_topics

noise_pct = round(100*(np.array(new_topics)==-1).mean(),1)
print(f"After outlier reduction: {len(set(new_topics))-1} topics, {noise_pct}% noise")

## Results

In [ ]:
topic_info = model.get_topic_info()
topic_info.head(20)

In [ ]:
## FINAL CLEAN STANDARDIZED EXPORT
# 1. Create a CLEAN Topic Label Map (No numbers, Space-separated, Title-case)
topic_info = model.get_topic_info()
label_map = {}
for _, row in topic_info.iterrows():
    t_id = row['Topic']
    raw_name = row['Name']
    # Remove the "0_" or "-1_" prefix and clean up underscores
    if "_" in raw_name:
        clean_name = raw_name.split("_", 1)[-1].replace("_", " ").title()
    else:
        clean_name = raw_name.title()
    label_map[t_id] = clean_name

df['topic_label'] = df['topic_id'].map(label_map)

# 2. Reorganize Columns
if 'ner_text' not in df.columns: df['ner_text'] = df['clean_text']
final_df = df.rename(columns={'title': 'raw_text'})

# 3. Selection (Exactly 5 columns)
cols_to_keep = ['raw_text', 'clean_text', 'ner_text', 'topic_id', 'topic_label']
final_df = final_df[cols_to_keep]

# 4. Save and Download
output_fn = "final_clustering_results.csv"
final_df.to_csv(output_fn, index=False)
print(f"Success! Exported {len(final_df)} titles with clean, unique labels.")
from google.colab import files
files.download(output_fn)